# Modelo 1: Regresión de panel — PIB per cápita (8.1.1) ~ Estrés hídrico (6.4.2)

Ajuste del modelo base de efectos fijos bidireccionales (país y año) sobre
los datos reales de `data/panel.db`, reutilizando `src/panel_base.py`
(Plan 03-01) sin modificarlo. Produce: comparación pooled/RE/FE, test de
Hausman, elección de errores estándar (clustered vs. Driscoll-Kraay) vía el
test de dependencia transversal de Pesaran, la comprobación de robustez
2000-2019 (sin años COVID, D-04), la serialización del modelo ajustado, y la
sección de limitaciones/amenazas a la validez (REPRO-03).


In [1]:
import sys
from pathlib import Path

# Notebook lives in notebook/, but `src/` and `data/panel.db` are relative to
# the project root -- add the project root to sys.path and chdir there so
# this notebook runs correctly regardless of launch method (nbconvert,
# Jupyter Lab, VS Code) or invocation cwd. Identical bootstrap pattern to
# notebook/2_1_construccion_panel_eda.ipynb.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pickle

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from linearmodels.panel import RandomEffects

from src import db, panel_base

DEP_VAR = "8.1.1"
INDEP_VARS = ["6.4.2"]

engine = db.get_engine(str(PROJECT_ROOT / "data" / "panel.db"))


In [2]:
# Full-table reads (no LIMIT) -- 03-RESEARCH.md Finding 3: a LIMIT-sampled
# read can misleadingly show object dtype on sparser indicator columns.
panel_clean = pd.read_sql("SELECT * FROM panel_clean", engine)
panel_exclusions = pd.read_sql("SELECT * FROM panel_exclusions", engine)

print("panel_clean shape:", panel_clean.shape)
print("panel_exclusions shape:", panel_exclusions.shape)


panel_clean shape: (4923, 12)
panel_exclusions shape: (529, 7)


In [3]:
filtered = panel_base.filter_by_exclusions(
    panel_clean, panel_exclusions, dep_var=DEP_VAR, indep_vars=INDEP_VARS
)

n_countries = filtered["country_code"].nunique()
n_obs = len(filtered)

# 03-RESEARCH.md Finding 4 expectation: ~171 countries, ~3,879 observations.
EXPECTED_COUNTRIES = 171
EXPECTED_OBS = 3879
country_drift = abs(n_countries - EXPECTED_COUNTRIES) / EXPECTED_COUNTRIES
obs_drift = abs(n_obs - EXPECTED_OBS) / EXPECTED_OBS

print(f"Countries after filter_by_exclusions: {n_countries} (expected ~{EXPECTED_COUNTRIES}, drift {country_drift:.1%})")
print(f"Observations after filter_by_exclusions: {n_obs} (expected ~{EXPECTED_OBS}, drift {obs_drift:.1%})")

if country_drift > 0.10 or obs_drift > 0.10:
    display(Markdown(
        f"**ALERTA:** la muestra se desvía más de un 10% de la expectativa de "
        f"03-RESEARCH.md Finding 4 (países: {country_drift:.1%}, observaciones: "
        f"{obs_drift:.1%}). Investigar antes de continuar -- no proceder "
        f"silenciosamente sobre una discrepancia grande."
    ))
else:
    display(Markdown(
        f"Tamaño muestral confirmado dentro del 10% de la expectativa de "
        f"Finding 4: **{n_countries} países**, **{n_obs} observaciones** "
        f"(países: {country_drift:.1%} de desvío, observaciones: {obs_drift:.1%} de desvío)."
    ))


Countries after filter_by_exclusions: 171 (expected ~171, drift 0.0%)
Observations after filter_by_exclusions: 3933 (expected ~3879, drift 1.4%)


Tamaño muestral confirmado dentro del 10% de la expectativa de Finding 4: **171 países**, **3933 observaciones** (países: 0.0% de desvío, observaciones: 1.4% de desvío).

## Ajuste provisional (primer paso) y elección del tipo de error estándar

Se ajusta primero el modelo base con `cov_type="clustered"` (por país) como
valor por defecto, únicamente para obtener los residuos necesarios para el
test de dependencia transversal de Pesaran. Este primer ajuste es
**provisional**: el tipo de error estándar definitivo se decide a partir del
resultado del test de Pesaran (`panel_base.choose_cov_type`), y el modelo se
reajusta a continuación con esa elección antes de usarse en el resto del
análisis (tabla de comparación, test de Hausman, comprobación de robustez).


In [4]:
provisional_fit = panel_base.fit_panel_model(
    filtered,
    dep_var=DEP_VAR,
    indep_vars=INDEP_VARS,
    entity_effects=True,
    time_effects=True,
    cov_type="clustered",
    cluster_entity=True,
)

pesaran_result = panel_base.pesaran_cd_test(provisional_fit.resids)
chosen_cov_type, chosen_cov_config = panel_base.choose_cov_type(pesaran_result)

reject_h0 = pesaran_result["pvalue"] < 0.05
display(Markdown(
    f"""**Test de dependencia transversal de Pesaran:** estadístico
CD = {pesaran_result['statistic']:.4f}, p-valor = {pesaran_result['pvalue']:.4f}.

{'Se **rechaza** H0 (no hay dependencia transversal)' if reject_h0 else 'No se rechaza H0 (no hay dependencia transversal)'}
al 5%, por lo que `choose_cov_type` selecciona
`cov_type="{chosen_cov_type}"` (`{chosen_cov_config}`) para el resto del
análisis (modelo base, test de Hausman, y comprobación de robustez).

**Matiz de interpretación (heredado del Plan 03-01):** un ajuste de efectos
fijos bidireccionales induce mecánicamente una pequeña correlación negativa
entre los residuos de distintos países (efecto conocido del *time-demeaning*,
De Hoyos & Sarafidis 2006), que puede empujar el test de Pesaran hacia el
rechazo de H0 incluso sin dependencia transversal genuina -- efecto más
notorio con N pequeño. Aquí N={filtered['country_code'].nunique()} países,
sustancialmente mayor que el caso sintético N=30 donde se documentó este
artefacto por primera vez, por lo que el efecto debería ser más débil; aun
así, si el test rechaza H0, este resultado se reporta con esta salvedad como
contexto, no como evidencia inequívoca de dependencia transversal real."""
))


C:\Users\olbap\source\repos\TFB\.venv\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


**Test de dependencia transversal de Pesaran:** estadístico
CD = 3.2042, p-valor = 0.0014.

Se **rechaza** H0 (no hay dependencia transversal)
al 5%, por lo que `choose_cov_type` selecciona
`cov_type="kernel"` (`{'kernel': 'bartlett'}`) para el resto del
análisis (modelo base, test de Hausman, y comprobación de robustez).

**Matiz de interpretación (heredado del Plan 03-01):** un ajuste de efectos
fijos bidireccionales induce mecánicamente una pequeña correlación negativa
entre los residuos de distintos países (efecto conocido del *time-demeaning*,
De Hoyos & Sarafidis 2006), que puede empujar el test de Pesaran hacia el
rechazo de H0 incluso sin dependencia transversal genuina -- efecto más
notorio con N pequeño. Aquí N=171 países,
sustancialmente mayor que el caso sintético N=30 donde se documentó este
artefacto por primera vez, por lo que el efecto debería ser más débil; aun
así, si el test rechaza H0, este resultado se reporta con esta salvedad como
contexto, no como evidencia inequívoca de dependencia transversal real.

In [5]:
final_fe_results = panel_base.fit_panel_model(
    filtered,
    dep_var=DEP_VAR,
    indep_vars=INDEP_VARS,
    entity_effects=True,
    time_effects=True,
    cov_type=chosen_cov_type,
    **chosen_cov_config,
)

print(final_fe_results.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  8.1.1   R-squared:                       1.3e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.0015
No. Observations:                3879   R-squared (Within):            6.779e-05
Date:                Sun, Jul 12 2026   R-squared (Overall):             -0.0003
Time:                        17:37:11   Log-likelihood                -1.182e+04
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      0.0479
Entities:                         171   P-value                           0.8268
Avg Obs:                       22.684   Distribution:                  F(1,3685)
Min Obs:                       17.000                                           
Max Obs:                       23.000   F-statistic (robust):             0.0496
                            

C:\Users\olbap\source\repos\TFB\.venv\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [6]:
comparison = panel_base.compare_specifications(filtered, dep_var=DEP_VAR, indep_vars=INDEP_VARS)
print(comparison)


C:\Users\olbap\source\repos\TFB\.venv\Lib\site-packages\linearmodels\panel\model.py:919: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
C:\Users\olbap\source\repos\TFB\.venv\Lib\site-packages\linearmodels\panel\model.py:2751: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


C:\Users\olbap\source\repos\TFB\.venv\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                            Model Comparison                           
                                Pooled                RE             FE
-----------------------------------------------------------------------
Dep. Variable                    8.1.1             8.1.1          8.1.1
Estimator                    PooledOLS     RandomEffects       PanelOLS
No. Observations                  3879              3879           3879
Cov. Est.                   Unadjusted        Unadjusted     Unadjusted
R-squared                       0.0045            0.0017        1.3e-05
R-Squared (Within)             -0.0006           -0.0002      6.779e-05
R-Squared (Between)             0.0429            0.0374        -0.0015
R-Squared (Overall)             0.0045            0.0042        -0.0003
F-statistic                     17.406            6.6700         0.0479
P-value (F-stat)                0.0000            0.0098         0.8268
=====================     ============   ===============   =====

In [7]:
# Separate RandomEffects fit for the Hausman test, using the SAME cov_type /
# cov_config the Pesaran test selected for final_fe_results -- NOT
# compare_specifications' internal unadjusted RE fit, and not a third,
# arbitrary choice (methodologically required: the Hausman formula assumes
# both covariance matrices are estimated the same way).
indexed_for_hausman = filtered.set_index(["country_code", "year"]).copy()
for col in [DEP_VAR, *INDEP_VARS]:
    indexed_for_hausman[col] = pd.to_numeric(indexed_for_hausman[col], errors="coerce")

hausman_re_results = RandomEffects(
    indexed_for_hausman[DEP_VAR], indexed_for_hausman[INDEP_VARS]
).fit(cov_type=chosen_cov_type, **chosen_cov_config)

hausman_result = panel_base.hausman_test(final_fe_results, hausman_re_results)

hausman_reject = hausman_result["pvalue"] < 0.05
display(Markdown(
    f"""**Test de Hausman (FE vs. RE):** estadístico
H = {hausman_result['statistic']:.4f} (df={hausman_result['df']}),
p-valor = {hausman_result['pvalue']:.4f}. Ambos lados del test (FE y RE) se
ajustaron con el mismo tipo de error estándar elegido arriba
(`cov_type="{chosen_cov_type}"`), condición necesaria para que la comparación
sea metodológicamente válida.

{'Se **rechaza** H0 (los estimadores FE y RE difieren sistemáticamente)' if hausman_reject else 'No se rechaza H0 (FE y RE no difieren significativamente en esta muestra)'}
al 5%. Independientemente de este resultado, la especificación de efectos
fijos (FE) es la elegida para el Modelo 1 por decisión ya fijada en D-03
(`03-CONTEXT.md`) -- el test de Hausman se reporta aquí como la
justificación documentada que exige el Criterio de Éxito #3 de la Fase 3,
no como un punto de decisión en vivo que pudiera cambiar el tipo de modelo."""
))


C:\Users\olbap\source\repos\TFB\.venv\Lib\site-packages\linearmodels\panel\model.py:2751: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


**Test de Hausman (FE vs. RE):** estadístico
H = 0.2566 (df=1),
p-valor = 0.6125. Ambos lados del test (FE y RE) se
ajustaron con el mismo tipo de error estándar elegido arriba
(`cov_type="kernel"`), condición necesaria para que la comparación
sea metodológicamente válida.

No se rechaza H0 (FE y RE no difieren significativamente en esta muestra)
al 5%. Independientemente de este resultado, la especificación de efectos
fijos (FE) es la elegida para el Modelo 1 por decisión ya fijada en D-03
(`03-CONTEXT.md`) -- el test de Hausman se reporta aquí como la
justificación documentada que exige el Criterio de Éxito #3 de la Fase 3,
no como un punto de decisión en vivo que pudiera cambiar el tipo de modelo.

In [8]:
models_dir = PROJECT_ROOT / "data" / "modelos"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "model1_gdp.pkl"
with open(model_path, "wb") as f:
    pickle.dump(final_fe_results, f)

print(f"Modelo serializado en: {model_path}")


Modelo serializado en: C:\Users\olbap\source\repos\TFB\data\modelos\model1_gdp.pkl


In [9]:
with open(model_path, "rb") as f:
    reloaded_results = pickle.load(f)

params_match = np.allclose(reloaded_results.params.values, final_fe_results.params.values)
print("Los parámetros del modelo recargado coinciden con el modelo en memoria:", params_match)
assert params_match, "El modelo recargado desde .pkl no coincide con el modelo ajustado en memoria"


Los parámetros del modelo recargado coinciden con el modelo en memoria: True


## Comprobación de robustez: submuestra 2000-2019 (excluyendo años COVID)

Per D-04 (`03-CONTEXT.md`), la comprobación de robustez de esta fase consiste
en reajustar el modelo base sobre la submuestra 2000-2019, excluyendo los
años 2020-2022 (shocks atípicos globales de la pandemia) -- **no** una
variante con controles adicionales (`6.4.1`+`8.2.1`); esa alternativa fue
considerada durante la discusión de fase y explícitamente no elegida, y no
debe asumirse como una segunda comprobación implícita.


In [10]:
robustness_df = filtered[filtered["year"] <= 2019]

robustness_fit = panel_base.fit_panel_model(
    robustness_df,
    dep_var=DEP_VAR,
    indep_vars=INDEP_VARS,
    entity_effects=True,
    time_effects=True,
    cov_type=chosen_cov_type,
    **chosen_cov_config,
)

print(robustness_fit.summary)


C:\Users\olbap\source\repos\TFB\.venv\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  8.1.1   R-squared:                        0.0001
Estimator:                   PanelOLS   R-squared (Between):             -0.0059
No. Observations:                3366   R-squared (Within):               0.0002
Date:                Sun, Jul 12 2026   R-squared (Overall):             -0.0016
Time:                        17:37:11   Log-likelihood                -1.007e+04
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      0.3880
Entities:                         171   P-value                           0.5334
Avg Obs:                       19.684   Distribution:                  F(1,3175)
Min Obs:                       14.000                                           
Max Obs:                       20.000   F-statistic (robust):             0.5234
                            

In [11]:
base_coef = final_fe_results.params[INDEP_VARS[0]]
base_pvalue = final_fe_results.pvalues[INDEP_VARS[0]]
robust_coef = robustness_fit.params[INDEP_VARS[0]]
robust_pvalue = robustness_fit.pvalues[INDEP_VARS[0]]

same_sign = (base_coef > 0) == (robust_coef > 0)
magnitude_ratio = abs(robust_coef / base_coef) if base_coef != 0 else float("nan")
both_significant = (base_pvalue < 0.05) and (robust_pvalue < 0.05)
both_not_significant = (base_pvalue >= 0.05) and (robust_pvalue >= 0.05)
same_significance_conclusion = both_significant or both_not_significant

qualitatively_consistent = same_sign and same_significance_conclusion

display(Markdown(
    f"""**Comparación base (2000-2022) vs. robustez (2000-2019, sin COVID):**

| | Coeficiente ({INDEP_VARS[0]}) | p-valor | Significativo (5%) |
|---|---|---|---|
| Modelo base | {base_coef:.5f} | {base_pvalue:.4f} | {'Sí' if base_pvalue < 0.05 else 'No'} |
| Robustez (2000-2019) | {robust_coef:.5f} | {robust_pvalue:.4f} | {'Sí' if robust_pvalue < 0.05 else 'No'} |

Signo igual: {'sí' if same_sign else 'no'}. Razón de magnitudes
(robustez/base): {magnitude_ratio:.2f}x.

**Veredicto:** el resultado es
{'**cualitativamente consistente**' if qualitatively_consistent else '**NO cualitativamente consistente**'}
entre la muestra completa y la submuestra sin años COVID -- el signo del
coeficiente {'se mantiene' if same_sign else 'cambia'} y la conclusión de
significancia estadística al 5% {'se mantiene' if same_significance_conclusion else 'cambia'}
al excluir 2020-2022."""
))


**Comparación base (2000-2022) vs. robustez (2000-2019, sin COVID):**

| | Coeficiente (6.4.2) | p-valor | Significativo (5%) |
|---|---|---|---|
| Modelo base | -0.00014 | 0.8238 | No |
| Robustez (2000-2019) | -0.00044 | 0.4694 | No |

Signo igual: sí. Razón de magnitudes
(robustez/base): 3.14x.

**Veredicto:** el resultado es
**cualitativamente consistente**
entre la muestra completa y la submuestra sin años COVID -- el signo del
coeficiente se mantiene y la conclusión de
significancia estadística al 5% se mantiene
al excluir 2020-2022.

## Limitaciones / Amenazas a la validez

**Causalidad inversa.** El coeficiente estimado sobre el estrés hídrico
(`6.4.2`) no debe interpretarse como evidencia de causalidad unidireccional
hacia el crecimiento del PIB per cápita. Es plausible -- y consistente con
el riesgo identificado en la propuesta oficial del TFB -- que la relación
opere también en sentido contrario: un peor desempeño económico puede
limitar la capacidad de un país para invertir en infraestructura hídrica
(desalinización, redes de distribución, tratamiento de aguas), agravando su
propio estrés hídrico medido. Los efectos fijos por país y por año absorben
heterogeneidad no observada constante en el tiempo (o común a todos los
países en un año dado), pero no resuelven por sí solos un problema de
causalidad inversa contemporánea entre las dos variables.

**Endogeneidad por variables omitidas.** Variables no incluidas en la
especificación -- calidad institucional/de gobernanza, exposición climática
general (más allá del estrés hídrico estrictamente medido por `6.4.2`),
shocks de política económica específicos de cada país -- podrían estar
correlacionadas simultáneamente con el estrés hídrico y con el crecimiento
del PIB per cápita. Los efectos fijos de país y año solo absorben la parte de
estas variables omitidas que es constante en el tiempo por país o común a
todos los países en un año dado; si varían dentro de un país a lo largo del
tiempo de forma correlacionada con el regresor de interés, el coeficiente
estimado puede reflejar en parte esa confusión, no un efecto causal limpio
del estrés hídrico.

**Conexión con el diseño del proyecto.** Por estas dos razones, el enfoque
de esta memoria (ver `PROJECT.md`) trata explícitamente cualquier simulación
contrafactual posterior (Fase 4: "cuánto mejoraría el PIB si el estrés
hídrico se redujera en X puntos porcentuales") como un **análisis de
sensibilidad**, no como una predicción causal -- el modelo de panel con
efectos fijos es la mejor aproximación disponible dentro del alcance de este
TFB para controlar heterogeneidad no observada, pero no elimina por completo
el riesgo de causalidad inversa ni de endogeneidad por variables omitidas
descrito arriba, y los resultados deben leerse con esa cautela ante el
tribunal.
